# Final Computational Neuroscience Project: Does Model Complexity Improve The Prediction of a Neuron's Spiking Based on LFP Data?
by Eitan Goldfein, Malakai, and Winnie


# Introduction


  ### Background
The hippocampus plays a key role in memory and spatial navigation, and its neurons show rich relationships with ongoing brain rhythms. In particular, local field potentials (LFPs) in hippocampal area CA1 contain prominent theta and gamma oscillations that organize the timing of neuronal spikes. Place cells, for example, tend to fire at specific phases of the theta cycle, and sequences of place cell activity can be “replayed” during rest and wheel-running.

At the same time, modern machine learning tools make it possible to ask not just whether spikes are related to LFP rhythms, but how well we can actually predict spiking activity from LFP signals. Classical models like Generalized Linear Models (GLMs) are relatively simple and interpretable, while neural networks (NNs) can, in principle, capture more complex nonlinear relationships. Our project uses the hippocampal dataset from class to directly compare these two approaches for predicting neuronal spike counts from LFP activity.

## Hypothesis


  Hypotehsis:  Model comparison: A neural network will achieve higher predictive accuracy than a GLM, because it can capture nonlinear relationships between LFP and spike counts.

## Data
In this dataset, a rat performs a Three-Arm Delayed Sequence Task, where each run down an arm is treated as a separate trial. During the task, local field potentials (LFPs) were recorded from electrodes in CA1 of the hippocampus, along with spike trains from many neurons. For our project, we focus on running data and use LFP-based signals as inputs to our models. These predictors are then fed into both the neural network and the GLM to see how well each model can predict the neurons’ spike counts.

## Variables
To predict CA1 spiking, we build a Toeplitz design matrix from the LFP: for each time bin, the input vector contains the recent LFP history (multiple time lags for each channel). These rows of the Toeplitz matrix are our predictors, and the outputs are the corresponding binned spike counts for a subset of neurons (often the top 20 most active).

We then fit two models on this same Toeplitz design matrix:

* A Poisson GLM, which assumes spike counts follow a Poisson process whose log
   firing rate is a linear function of the LFP history.


  

*   A neural network, which takes the same LFP history as input but can learn
    nonlinear mappings from LFP to spike counts.

## Hyperparameters

We adjust a few main hyperparameters in our project:

**Design / data setup**
- Number of LFP time lags in the Toeplitz matrix   
- Train–test split (for example, 80% train and 20% test)  

**Poisson GLM**
- Regularization strength (how strongly we penalize large weights)  
- Maximum number of optimization iterations
- Batch Size(Pytorch)
- Learning Rate(Pytorch)
- Number Epochs(Pytorch)

**Neural network**
- Number of hidden layers and number of units per layer  
- Learning rate  
- Number of training epochs  
- Batch size  
- Dropout rate (if we use hidden layers)

We choose these hyperparameters by performing a hyperparamter search using OPTUNA

# Python Dependencies

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as sk
import scipy.io

import torch as torch
import torch.nn as nn

from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split

from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import r2_score



# Data Cleaning and Processing

In [ ]:
!pip install gdown
!gdown --id 1dHlYOUBMyIIPP8it63DGMyaBASEim61S

mat_data = scipy.io.loadmat('data03.mat')
lfps = pd.DataFrame(mat_data['lfps'])
spikes = pd.DataFrame(mat_data['spikes'])
lapID = pd.DataFrame(mat_data['lapID'])

### Toeplitz Construction
In this section we construct a Toeplitz-style design matrix from the LFPs.
For each time bin, we include LFP values from a window of past and future time lags around that bin.
This creates a feature vector that captures short-term temporal context in the LFP signal, which is
important because spikes depend not just on the instantaneous LFP but on its recent history as well.
We also  use the lapID labels to
filter the data down to “running” periods only. This gives us a clean subset of time bins where the
rat is actively moving, which is the behavioral regime we focus on for predicting spikes.


In [ ]:
# Select only laps where rat is "running"
#Here lapID(column 3), 1 or 3 encodes the running state
mask= lapID[3].isin([1,3])

#Apply mask to keep only running time points
lapID_filter = lapID[mask]
running_lfp = lfps.loc[mask.values,:]
running_spikes = spikes.loc[mask.values,:]

#Filter for times where the rat is only running as to get more active data
#We now create a Toeplitz design matrix that includes time-lagged LFPs
#We have 5 times the orignial number of channeles or "electrodes"

def create_toeplitz(df, time_lags):
  toeplitz_matrix = pd.DataFrame()
  for i in range(-time_lags, time_lags + 1):
    #Shift the entire time seires by i steps and the edges are filled with 0
    #0 is the neutral based that avoids NaN values
      tempMatrix = df.shift(i).fillna(0)
      toeplitz_matrix = pd.concat([toeplitz_matrix, tempMatrix], axis=1)

  # Apply the mask to the *fully constructed* Toeplitz matrix so that inputs and outputs align
  return toeplitz_matrix.loc[mask.values,:]
# Build the final design matrix from all LFPs with ±2 time lags.
DesignMatrix = create_toeplitz(lfps, 2)
DesignMatrix

## Model Input Prep
Here we convert the design matrix and spike counts into the input (`X`) and output (`y`) arrays used by
our models. We split the data into a training set and a held-out test set, keeping `shuffle=False` so
that the temporal order of the experiment is preserved (the test set comes from later time bins).
We then use `RobustScaler` to standardize the inputs in a way that is less sensitive to outliers, and
finally convert the arrays into PyTorch tensors and wrap them in a `DataLoader` so that we can train
our models in mini-batches.

In [ ]:
#Convert to basic nump array for sklearn and for scaling
X = DesignMatrix.values       #Inputs: LFP Toeplitz Features
y = running_spikes.values     #Outputs : spike counts for 403 Neurons


#Test/train split where 20# of data is held out for evaluation.
# Random sate = 0 makes it a reproducable split
# Shuffle = Maintains temporal order of the experiment
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, shuffle=False
)
#Scales inputs with RobustScaler, which is a standerdizer that is robust to outliers
scaler = RobustScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

#Converts to PyTorch tensors for NN and Pytorch-GLM
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

#DataLeader: We also keep shuffle as False so batches respect time order
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=False)


# Neural Network


### Neural Network Architecture
This section defines our main nonlinear model: a feedforward neural network that predicts the spike
counts of 403 neurons from the LFP Toeplitz features. The network has two hidden layers with ReLU
nonlinearities and dropout, followed by a Softplus output layer to ensure positive firing rates.
We train this model using a Poisson negative log-likelihood loss, which is appropriate for count data,
and the AdamW optimizer, which provides stable training with built-in L2-style regularization.

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.net = nn.Sequential(
            #Input Layer # 960 Features -> 128 Hidden nodes
            nn.Linear(960, 128),
            nn.ReLU(),            #Non-Linearity
            nn.Dropout(0.104),    # ~ 10% dropout : regularization

            # Second hidden layer: 128 → 512 units.
            # Wider layer lets the network learn more complex combinations
            # of features after a smaller "bottleneck" layer.
            nn.Linear(128, 512),
            nn.ReLU(),
            nn.Dropout(0.496),   # ~50% dropout which helps prevent overfitting in a big layer

            # Output layer: 512 → 403 neurons (one rate per neuron).
            nn.Linear(512, 403),
            #Softplus ensures strictly positive firing rates, which fits the Poisson assumption
            nn.Softplus()
        )

    def forward(self, x):
        return self.net(x)

model = NeuralNet()

# Loss: Poisson NLL, appropriate for count data.
# log_input=False because the network outputs rates λ directly (not log λ).
#Full = true keeps loss numerically closer to real NLL. Not super signifiant decision.
loss_function = torch.nn.PoissonNLLLoss(log_input=False, full=True)

## Optimizer: AdamW (Adam with decoupled weight decay - essential acts as a form of L2 regularization)
# lr=5.56e-4 is relatively small for stability on dataset with some noise
optimizer = torch.optim.AdamW(model.parameters(), lr=0.000556)


## Training Loop
Here we train the neural network on the training data. We iterate over the data in mini-batches of 32
samples, computing the Poisson loss, backpropagating gradients, and updating the weights with AdamW.
Every 10 epochs we evaluate the model on both the training and test sets, tracking how the Poisson loss
evolves over time. This allows us to monitor convergence and check for signs of overfitting.

In [ ]:
epochs = 100              #100 passes over dataset to reach convergence
train_losses = []         #Store train and test loss every 10 Epochs
test_losses = []

for epoch in range(epochs):
    model.train()             #Putting network into training mode
    epoch_loss = 0

  #Gradient Descent Section
  # Train in mini-batches of 32 for efficient, stable updates,
  # keeping shuffle=False so batches follow the original time order.
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()           #Clears gradients from prior pass
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)    #Poisson Loss
        loss.backward()                           #BackPropagation Step
        optimizer.step()                          #Updates Wights
        epoch_loss += loss.item()

    # Evaluate on full train/test sets every 10 epochs to track overfitting
    if (epoch + 1) % 10 == 0:
        model.eval()
        with torch.no_grad():
            train_loss = loss_function(model(X_train), y_train).item()
            test_loss = loss_function(model(X_test), y_test).item()
            train_losses.append(train_loss)
            test_losses.append(test_loss)
        print(f"Epoch {epoch+1}/{epochs} | train loss {train_loss:.4f} | test loss {test_loss:.4f}")

# GLM
In this section we define a PyTorch implementation of a Poisson Generalized Linear Model (GLM).
Architecturally, this model is just a single linear model followed by an exponential nonlinearity,
which corresponds to a standard Poisson GLM with a log link. This gives us a simpler, linear baseline
to compare against the more flexible neural network while keeping the loss function and input features
the same.

## GLM Architecture (Neural Net with no Hidden Layers)

In [ ]:
class GLM(nn.Module):
    def __init__(self, input_size=960, output_size=403):
        super(GLM, self).__init__()
        # Single linear layer: this is basically a linear Poisson GLM.
        self.linear = nn.Linear(input_size, output_size)
    def forward(self, x):
        x = self.linear(x)
        # Exponential nonlinearity: λ = exp(Xβ)
        return torch.exp(x)

# Set GLM with the same input and output dimensions as NN
glm_model = GLM(input_size=960, output_size=403)


#Larger LR than NN because it is a simpler, "sturdier" model that can handle larger steps or "Jumps"
optimizer = torch.optim.AdamW(
    glm_model.parameters(),
    lr=0.0014179110224296831,
    weight_decay=0.0010953656938654556       #L2 regularization via Weight Decay
)


# Same Poisson NLL loss as before
lossFunction = torch.nn.PoissonNLLLoss(log_input=False, full=True)


## Training Loop(GLM)
We train the PyTorch GLM using the same training data and Poisson loss as the neural network.
Because the GLM is much simpler (one linear layer), we can use a slightly larger learning rate with
AdamW. As with the NN, we iterate in mini-batches and periodically compute train and test loss, which
lets us compare how the linear GLM fits the data relative to the deeper nonlinear model.


In [ ]:
epochs = 100
for epoch in range(epochs):
    glm_model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = glm_model(batch_X)
        loss = lossFunction(outputs, batch_y)
        loss.backward()
        optimizer.step()
#Same Idea as before with NN
    if (epoch + 1) % 20 == 0:
        glm_model.eval()
        with torch.no_grad():
            train_loss = lossFunction(glm_model(X_train), y_train).item()
            test_loss = lossFunction(glm_model(X_test), y_test).item()
        print(f"Epoch {epoch+1}/{epochs} | train loss {train_loss:.4f} | test loss {test_loss:.4f}")

## SKLearn GLM
This section introduces a second GLM baseline using sklearn’s `PoissonRegressor`.
We first prepare numpy versions of the training and test data, then fit a separate Poisson regression
for each neuron using `MultiOutputRegressor`. The sklearn GLM uses a likelihood-based optimizer and
an L2 regularization parameter (`alpha`) tuned to control overfitting. Comparing this implementation
to the PyTorch GLM helps us confirm that our results are not an artifact of a particular framework.


In [ ]:
# Right after the train/test split & scaling, BEFORE you convert to torch:
X_train_glm = scaler.fit_transform(X_train)   # scaled features for GLM
X_test_glm  = scaler.transform(X_test)
y_train_glm = y_train                         # spike counts
y_test_glm  = y_test

# Then convert copies for the NN if you want:
X_train_nn = torch.tensor(X_train_glm, dtype=torch.float32)
X_test_nn  = torch.tensor(X_test_glm, dtype=torch.float32)
y_train_nn = torch.tensor(y_train_glm, dtype=torch.float32)
y_test_nn  = torch.tensor(y_test_glm, dtype=torch.float32)


In [ ]:
from sklearn.linear_model import PoissonRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_poisson_deviance
import numpy as np


#Base Poisson Regressor (One Neuron)
base_poisson_glm = PoissonRegressor(
    alpha = 0.006051, # L2 Regularization Strength - tuned to control for overfitting
    max_iter = 900, fit_intercept = True, tol = 0.000524)


# Wrap in MultiOutputRegressor so we fit 403 independent GLMs in parallel, one for each neuron.
glm_all_neurons = MultiOutputRegressor(base_poisson_glm, n_jobs = -1)

#Fits on Same Train Data as the NN
glm_all_neurons.fit(X_train_glm , y_train_glm)

print("Finished fitting GLM for all neurons.")


# Model Analysis/Comparison
Finally, we evaluate and compare all three models: the nonlinear neural network, the PyTorch GLM, and
the sklearn GLM. For each neuron, we compute the Pearson correlation coefficient between the model’s
predicted firing rate and the true spike counts on held-out test data. We then plot correlation versus
mean firing rate for each model to see how performance depends on firing rate, and create scatter plots
comparing NN and GLM correlations neuron-by-neuron. These analyses let us directly assess whether the
more complex neural network provides a meaningful improvement over simpler GLMs for spike prediction.


## NN: correlation vs mean firing rate
This figure shows that the neural network generally predicts spikes better for neurons with higher mean firing rates: correlation tends to increase as firing rate goes up, while many very low–firing neurons remain hard to predict. This pattern is expected in spike train modeling, because neurons that fire rarely provide less information for the model to learn from and are more affected by sampling noise. Overall, the plot suggests that the nonlinear NN is able to capture meaningful structure in the activity of moderately and highly active neurons.

In [ ]:
# NN: correlation vs mean firing rate
model.eval()
with torch.no_grad():
    nn_pred = model(X_test)

# move tensors to CPU and convert to NumPy for correlation calculations
nn_pred_np = nn_pred.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

n_neurons = y_test_np.shape[1]
corr_nn = np.zeros(n_neurons)

for i in range(n_neurons):
    corr_nn[i] = np.corrcoef(nn_pred_np[:, i], y_test_np[:, i])[0, 1]

firing_rates_nn = y_test_np.mean(axis=0)
# Plot NN performance per neuron:
# x-axis = mean firing rate, y-axis = correlation (NN vs true spikes).
# Shows how prediction accuracy changes across low- vs high-firing neurons.
plt.figure(figsize=(6, 4))
plt.scatter(np.log(firing_rates_nn), corr_nn, s=8)
plt.xlabel("Log mean firing rate (spikes/bin)")
plt.ylabel("Correlation coefficient")
#plt.title("Neural Network: correlation vs mean firing rate")
plt.grid(True, alpha=0.3)
plt.show()

print("Average NN correlation:", np.nanmean(corr_nn))


## PyTorch GLM: correlation vs mean firing rate
For the PyTorch GLM, we see the same basic trend that higher–firing neurons are easier to model, but the correlations are generally lower than for the neural network. This indicates that a purely linear Poisson GLM can predict spike counts, especially for the most active cells, but it fails to capture as much of the moment-to-moment structure in the responses as the nonlinear NN. The spread of points near low firing rates also highlights that many neurons are only weakly predictable with a linear model.

In [ ]:
# PyTorch GLM: correlation vs mean firing rate
glm_model.eval()
with torch.no_grad():
    glm_pred = glm_model(X_test)

glm_pred_np = glm_pred.cpu().numpy()
y_test_np   = y_test.cpu().numpy()

n_neurons = y_test_np.shape[1]
corr_glm_torch = np.zeros(n_neurons)

for i in range(n_neurons):
    corr_glm_torch[i] = np.corrcoef(glm_pred_np[:, i], y_test_np[:, i])[0, 1]

firing_rates_glm_torch = y_test_np.mean(axis=0)

plt.figure(figsize=(6, 4))
plt.scatter(np.log(firing_rates_glm_torch), corr_glm_torch, s = 8)
plt.xlabel("Log mean firing rate (spikes/bin)")
plt.ylabel("Correlation coefficient")
#plt.title("GLM: correlation vs mean firing rate")
plt.grid(True, alpha=0.3)
plt.show()

print("Average PyTorch GLM correlation:", np.nanmean(corr_glm_torch))


## sklearn GLM: correlation vs mean firing rate
The sklearn GLM shows a very similar dependence on firing rate as the PyTorch GLM, with per-neuron correlations largely in the same range and following the same pattern across neurons. Any gap between the GLMs and the neural network can be interpreted as a true difference in model class.

In [ ]:
# sklearn GLM: correlation vs mean firing rate
# (uses X_test_glm, y_test_glm from the numpy/scaler pipeline)
glm_pred_sklearn = glm_all_neurons.predict(X_test_glm)
y_test_glm_np    = y_test_glm

n_neurons = y_test_glm_np.shape[1]
corr_glm_sklearn = np.zeros(n_neurons)

for i in range(n_neurons):
    corr_glm_sklearn[i] = np.corrcoef(glm_pred_sklearn[:, i],
                                      y_test_glm_np[:, i])[0, 1]

firing_rates_glm_sklearn = y_test_glm_np.mean(axis=0)

plt.figure(figsize=(6, 4))
plt.scatter(np.log(firing_rates_glm_sklearn), corr_glm_sklearn, s=3)
plt.xlabel("Log mean firing rate (test)")
plt.ylabel("Correlation coefficient")
plt.title("sklearn GLM: correlation vs mean firing rate")
plt.grid(True, alpha=0.3)
plt.show()

print("Average sklearn GLM correlation:", np.nanmean(corr_glm_sklearn))


## NN vs PyTorch GLM: per-neuron correlation scatter
In this scatter plot, each point is one neuron, with GLM correlation on the x-axis and NN correlation on the y-axis. The fact that most points lie above the diagonal line means that, for the majority of neurons, the neural network achieves higher prediction accuracy than the GLM on held-out data. This pattern is in line with our hypothesis that the nonlinear NN would outperform a Poisson GLM, especially for neurons whose firing is not well captured by a purely linear model.

In [ ]:
# Scatter of per-neuron correlations: NN vs PyTorch GLM
plt.figure(figsize=(6, 4))
plt.scatter(corr_glm_torch, corr_nn, s=5)

min_val = min(np.nanmin(corr_glm_torch), np.nanmin(corr_nn))
max_val = max(np.nanmax(corr_glm_torch), np.nanmax(corr_nn))
plt.plot([0, 1], [ 0, 1], linestyle='--')

plt.xlabel("GLM correlation")
plt.ylabel("NN correlation")
plt.title("Per-neuron correlation: NN vs GLM")
plt.grid(True, alpha=0.3)
plt.show()


# HyperParamter Search for NN and NN with no Hidden Layers

NN With Hidden Layers

In [ ]:
import optuna
import torch as torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Define a flexible neural network class
#class TunableNeuralNet(nn.Module): #Uncomment this line to run code
    def __init__(self, input_size, output_size, hidden_sizes, dropout_rates):
        super(TunableNeuralNet, self).__init__()

        layers = []
        prev_size = input_size

        # Build hidden layers
        for hidden_size, dropout in zip(hidden_sizes, dropout_rates):
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_size = hidden_size

        # Output layer
        layers.append(nn.Linear(prev_size, output_size))
        layers.append(nn.Softplus())

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    n_layers = trial.suggest_int('n_layers', 1, 4)
    hidden_sizes = []
    dropout_rates = []

    for i in range(n_layers):
        # Suggest hidden layer size (powers of 2 are common)
        hidden_size = trial.suggest_categorical(f'hidden_size_{i}', [64, 128, 256, 512, 768, 1024])
        hidden_sizes.append(hidden_size)

        # Suggest dropout rate
        dropout = trial.suggest_float(f'dropout_{i}', 0.1, 0.5)
        dropout_rates.append(dropout)

    # Suggest learning rate (log scale)
    lr = trial.suggest_float('lr', 1e-5, 1e-2, log=True)

    # Suggest optimizer
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'SGD'])

    # Suggest batch size
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256])

    # Create the model
    input_size = X_train.shape[1]  # 960
    output_size = y_train.shape[1]  # 403
    model = TunableNeuralNet(input_size, output_size, hidden_sizes, dropout_rates)

    # Create optimizer
    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name == 'AdamW':
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)

    # Create data loaders
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Loss function
    loss_function = torch.nn.PoissonNLLLoss(log_input=False, full=True)

    # Training loop
    n_epochs = 50  # Reduced for faster tuning
    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = loss_function(outputs, batch_y)
            loss.backward()
            optimizer.step()

        # Prune trial if it's not promising (optional but recommended)
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_loss = loss_function(model(X_test), y_test).item()

            trial.report(val_loss, epoch)

            # Handle pruning based on the intermediate value
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    # Final evaluation on validation set
    model.eval()
    with torch.no_grad():
        val_loss = loss_function(model(X_test), y_test).item()

    return val_loss


# Create and run the study
study = optuna.create_study(
    direction='minimize',  # We want to minimize validation loss
    pruner=optuna.pruners.MedianPruner()  # Prunes unpromising trials early
)

# Run optimization
study.optimize(objective, n_trials=50)  # Try 50 different hyperparameter combinations

# Print results
print("\n" + "="*50)
print("Best trial:")
print(f"  Value (validation loss): {study.best_trial.value:.4f}")
print(f"  Params: ")
for key, value in study.best_trial.params.items():
    print(f"    {key}: {value}")

# Visualize the optimization (optional)
try:
    import optuna.visualization as vis

    # Plot optimization history
    fig1 = vis.plot_optimization_history(study)
    fig1.show()

    # Plot parameter importances
    fig2 = vis.plot_param_importances(study)
    fig2.show()

    # Plot slice plot
    fig3 = vis.plot_slice(study)
    fig3.show()
except:
    print("\nInstall plotly for visualizations: pip install plotly")

NN With No Hidden Layers

In [ ]:
# Define a GLM as a PyTorch model (linear model with no hidden layers)
#class GLM(nn.Module): #uncomment this line to run
    def __init__(self, input_size=960, output_size=403):
        super(GLM, self).__init__()

        self.linear = nn.Linear(input_size, output_size)
    def forward(self, x):
        x = self.linear(x)

        return torch.exp(x)


# Define objective function for PyTorch GLM
def pytorch_glm_objective(trial):
    # Suggest hyperparameters
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)

    # Suggest optimizer
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'AdamW', 'SGD'])

    # Suggest weight decay (L2 regularization)
    weight_decay = trial.suggest_float('weight_decay', 1e-6, 1e-1, log=True)

    # Suggest batch size
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 128, 256, 512])

    # Create model
    model = GLM(input_size=X_train.shape[1], output_size=y_train.shape[1])

    # Create optimizer with weight decay (L2 regularization)
    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif optimizer_name == 'AdamW':
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay, momentum=0.9)

    # Create data loader
    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Loss function
    loss_function = torch.nn.PoissonNLLLoss(log_input=False, full=True)

    # Training loop (fewer epochs for GLM since it's simpler)
    n_epochs = 30
    for epoch in range(n_epochs):
        model.train()
        for batch_X, batch_y in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = loss_function(outputs, batch_y)
            loss.backward()
            optimizer.step()

        # Early stopping check
        if epoch % 10 == 0:
            model.eval()
            with torch.no_grad():
                val_loss = loss_function(model(X_test), y_test).item()

            trial.report(val_loss, epoch)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_loss = loss_function(model(X_test), y_test).item()

    return val_loss


# Create and run the study
pytorch_glm_study = optuna.create_study(
    direction='minimize',
    pruner=optuna.pruners.MedianPruner()
)

print("Starting PyTorch GLM hyperparameter search...")
pytorch_glm_study.optimize(pytorch_glm_objective, n_trials=40)

# Print results
print("\n" + "="*50)
print("Best PyTorch GLM trial:")
print(f"  Value (validation loss): {pytorch_glm_study.best_trial.value:.4f}")
print(f"  Params: ")
for key, value in pytorch_glm_study.best_trial.params.items():
    print(f"    {key}: {value}")

# Visualize results
try:
    import optuna.visualization as vis

    fig1 = vis.plot_optimization_history(pytorch_glm_study)
    fig1.show()

    fig2 = vis.plot_param_importances(pytorch_glm_study)
    fig2.show()
except:
    print("Install plotly for visualizations")

# Final Takeaway
Across all of our analyses, our main takeaway is that the nonlinear neural network consistently outperforms both Poisson GLMs in predicting hippocampal spike trains from LFP-based Toeplitz features, especially for moderately and highly active neurons. The correlation–versus–firing-rate plots show that all models struggle on very low–firing cells, but the NN achieves noticeably higher correlations for neurons with more robust activity, while the GLMs plateau earlier. The NN–vs–GLM scatter further confirms this: most neurons lie above the identity line, indicating better NN performance on a per-neuron basis, whereas the two GLM implementations largely agree with each other. Overall, these results are in line with our hypothesis that adding nonlinear structure allows the model to capture spike–LFP relationships that a simple linear Poisson GLM misses, suggesting that neural networks can provide a meaningful boost over classical GLMs for this kind of spike prediction task.

# Bibliography
* Benjamin, A. S., Fernandes, H. L., Tomlinson, T., Ramkumar, P., Ver Steeg, C., Chowdhury, R. H., Miller, L. E., & Kording, K. P. (2018). Modern machine learning as a benchmark for fitting neural responses. Frontiers in Computational Neuroscience, 12, 56.

We used this paper as our main motivation that modern machine-learning models (like neural networks) can serve as strong benchmarks for predicting neural responses compared to classical models.

* Gautam Agarwal et al., Spatially Distributed Local Fields in the Hippocampus Encode Rat Position. Science 344, 626-630 (2014). DOI:10.1126/science.1250444

This paper uses LFPs in hippocampus to decode rat position. This is the dataset used in this project

* KordingLab. (2018– ). spykesML: Machine learning algorithms for spike prediction. GitHub repository. Available at: github.com/KordingLab/spykesML

We drew on this repository for example implementations and best practices for spike prediction models, and as a reference point for our own NN–vs–GLM comparison.


* Zhou, P., Burton, S. D., Snyder, A. C., Smith, M. A., Urban, N. N., & Kass, R. E. (2015). Establishing a statistical link between network oscillations and neural synchrony. PLOS Computational Biology, 11(10), e1004549

This work links network oscillations and neural synchrony, providing background for why oscillatory LFP structure should be related to coordinated spiking activity in our dataset.

